## Previous Application Feature Engineering

In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [2]:
from src.data_utils import load_raw, save_interim
import numpy
import pandas

In [4]:
df_prev=load_raw("previous_application.csv")

### Contract Status Aggregation

In [5]:
prev=df_prev.copy()
contract_status=df_prev.groupby(
    ['SK_ID_CURR', 'NAME_CONTRACT_STATUS'])['SK_ID_PREV'].size().unstack(fill_value=0)

contract_status['PREV_TOTAL_APL']=contract_status.sum(axis=1)
contract_status['PREV_APPROVED_COUNT']=contract_status.get('Approved', 0)
contract_status['PREV_REFUSAL_COUNT']=contract_status.get('Refused', 0)

contract_status['PREV_APPROVAL_RATE']=(
    contract_status.get('Approved', 0) / contract_status['PREV_TOTAL_APL']
)
contract_status['PREV_REFUSAL_RATE']=(
    contract_status.get('Refused', 0) / contract_status['PREV_TOTAL_APL']
)

contract_status=contract_status.reset_index()

### Amount Based Aggregation

In [6]:
amount_agg=prev.groupby('SK_ID_CURR').agg(
    PREV_AVG_APPLICATION = ('AMT_APPLICATION', 'mean'),
    PREV_AVG_ANNUITY = ('AMT_ANNUITY', 'mean'),
    PREV_AVG_CREDIT = ('AMT_CREDIT', 'mean'),
    PREV_MAX_CREDIT = ('AMT_CREDIT', 'max')
)

### Decision Recency Features

In [7]:
prev['DAYS_DECISION_POS']=-prev['DAYS_DECISION']

days_agg=prev.groupby('SK_ID_CURR').agg(
    PREV_AVG_DAYS_DECISION = ('DAYS_DECISION_POS', 'mean'),
    PREV_MIN_DAYS_DECISION = ('DAYS_DECISION_POS', 'min')
)

### Merge and Save

In [9]:
prev_agg = (
    contract_status
    .merge(amount_agg, on='SK_ID_CURR', how='left')
    .merge(days_agg, on='SK_ID_CURR', how='left')
)

In [10]:
col = ['SK_ID_CURR',
    'PREV_TOTAL_APL',
    'PREV_APPROVED_COUNT',
    'PREV_REFUSAL_COUNT',
    'PREV_APPROVAL_RATE',
    'PREV_REFUSAL_RATE',
    'PREV_AVG_APPLICATION',
    'PREV_AVG_ANNUITY',
    'PREV_AVG_CREDIT',
    'PREV_MAX_CREDIT',
    'PREV_AVG_DAYS_DECISION',
    'PREV_MIN_DAYS_DECISION']

prev_agg = prev_agg[col].reset_index(drop=True)

In [11]:
save_interim(prev_agg, 'previous_application_agg.csv')

In [37]:
print(prev_agg.columns)

Index(['SK_ID_CURR', 'PREV_TOTAL_APL', 'PREV_APPROVED_COUNT',
       'PREV_REFUSAL_COUNT', 'PREV_APPROVAL_RATE', 'PREV_REFUSAL_RATE',
       'PREV_AVG_APPLICATION', 'PREV_AVG_ANNUITY', 'PREV_AVG_CREDIT',
       'PREV_MAX_CREDIT', 'PREV_AVG_DAYS_DECISION', 'PREV_MIN_DAYS_DECISION'],
      dtype='object')
